In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!git clone https://github.com/muhammad-omer-dev/DPO_PA2.git
# ! git pull origin

Cloning into 'DPO_PA2'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 3), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 249.27 KiB | 24.93 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
%cd /kaggle/working/DPO_PA2

/kaggle/working/DPO_PA2


In [4]:
!ls

data  eval_sets       README.md		submission_adapter.py
docs  pyproject.toml  requirements.txt	tests


In [5]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ..

In [6]:
!nvidia-smi

import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Visible GPUs:", torch.cuda.device_count())

Mon Oct 12 06:56:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

.......                                                                  [100%]
7 passed in 2.06s


In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
import random
import math
import gc
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import get_cosine_schedule_with_warmup

from submission_adapter import (
    build_sft_batch,
    compute_response_logprobs,
    compute_sft_loss,
    build_dpo_batch,
    compute_dpo_loss_from_logps,
    compute_dpo_loss,
)



SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
print("Visible GPUs:", torch.cuda.device_count())
os.makedirs("outputs/eval", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

Visible GPUs: 1


In [9]:
!python -m pip install -q "trl==0.26.0" peft
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer
print("TRL imported")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 10.9 MB/s eta 0:00:00
TRL imported


In [10]:
ALPACA_EVAL_URL = (
    "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/"
    "alpaca_eval.json"
)

alpaca_data = load_dataset("json", data_files=ALPACA_EVAL_URL, split="train")
alpaca_rows = list(alpaca_data)

random.seed(SEED)
random.shuffle(alpaca_rows)
alpaca_rows = alpaca_rows[:300]

alpaca_subset = []
for index, row in enumerate(alpaca_rows):
    alpaca_subset.append({
        "id": "alpaca-eval-42-" + str(index).zfill(4),
        "instruction": row["instruction"],
        "dataset": "alpaca_eval",
    })

with open("eval_sets/alpaca_eval_300_seed42.json", "w", encoding="utf-8") as file:
    json.dump(alpaca_subset, file, indent=2, ensure_ascii=False)

print("Saved", len(alpaca_subset), "AlpacaEval prompts")

alpaca_eval.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Saved 300 AlpacaEval prompts


In [11]:
# def generate_qwen_responses(model, tokenizer, rows, output_path, generator_name, limit=None):
#     device = next(model.parameters()).device
#     results = []

#     selected_rows = rows
#     if limit is not None:
#         selected_rows = rows[:limit]

#     for index, row in enumerate(selected_rows):
#         prompt = "### Instruction:\n" + row["instruction"].strip() + "\n\n### Response:\n"
#         encoded = tokenizer(prompt, return_tensors="pt")
#         input_ids = encoded.input_ids.to(device)
#         attention_mask = encoded.attention_mask.to(device)

#         with torch.no_grad():
#             generated = model.generate(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 max_new_tokens=256,
#                 do_sample=False,
#                 pad_token_id=tokenizer.pad_token_id,
#                 eos_token_id=tokenizer.eos_token_id,
#             )

#         new_tokens = generated[0][input_ids.shape[1]:]
#         output_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

#         results.append({
#             "id": row.get("id", str(index)),
#             "instruction": row["instruction"],
#             "output": output_text,
#             "generator": generator_name,
#             "dataset": row.get("dataset", "unknown"),
#         })

#     with open(output_path, "w", encoding="utf-8") as file:
#         json.dump(results, file, indent=2, ensure_ascii=False)

#     print("Saved", len(results), "responses to", output_path)
#     return results

In [12]:
# BASE_MODEL = "Qwen/Qwen2.5-0.5B"
# base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# if base_tokenizer.pad_token_id is None:
#     base_tokenizer.pad_token = base_tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     dtype=torch.float16,
# ).to("cuda:0")
# base_model.eval()

# base_sanity = generate_qwen_responses(
#     base_model,
#     base_tokenizer,
#     alpaca_subset,
#     "outputs/eval/base_sanity_3.json",
#     "qwen2.5-0.5b-base",
#     limit=3,
# )

# for row in base_sanity:
#     print("INSTRUCTION:", row["instruction"])
#     print("OUTPUT:", row["output"])
#     print("-" * 80)

In [13]:
# generate_qwen_responses(
#     base_model,
#     base_tokenizer,
#     alpaca_subset,
#     "outputs/eval/base_alpaca.json",
#     "qwen2.5-0.5b-base",
# )

# del base_model
# del base_tokenizer
# gc.collect()
# torch.cuda.empty_cache()

In [14]:
# SFT_MODEL = "Qwen/Qwen2.5-0.5B"
# SFT_TRAIN_EXAMPLES = 30000
# SFT_VALIDATION_EXAMPLES = 1000
# SFT_MAX_LENGTH = 512
# SFT_BATCH_SIZE = 2
# SFT_GRAD_ACCUM = 16
# SFT_LEARNING_RATE = 2e-5
# SFT_WARMUP_RATIO = 0.03
# SFT_WEIGHT_DECAY = 0.1
# SFT_GRAD_CLIP = 1.0
# SFT_EPOCHS = 1
# SFT_OUTPUT = "outputs/sft"

# os.makedirs(SFT_OUTPUT, exist_ok=True)
# SFT_METRICS = os.path.join(SFT_OUTPUT, "metrics.jsonl")
# if os.path.exists(SFT_METRICS):
#     os.remove(SFT_METRICS)

In [15]:
# def make_sft_examples(dataset, number_of_examples):
#     examples = []

#     for index in range(number_of_examples):
#         row = dataset[index]
#         instruction = row["messages"][0]["content"]
#         response = row["messages"][1]["content"]

#         examples.append({
#             "prompt": "### Instruction:\n" + instruction.strip() + "\n\n### Response:\n",
#             "response": response,
#         })

#     return examples


# def save_jsonl_metric(path, metric):
#     with open(path, "a", encoding="utf-8") as file:
#         file.write(json.dumps(metric) + "\n")


# def get_sft_validation_loss(model, tokenizer, examples, device):
#     model.eval()
#     total_negative_logp = 0.0
#     total_response_tokens = 0

#     for start in range(0, len(examples), SFT_BATCH_SIZE):
#         rows = examples[start:start + SFT_BATCH_SIZE]
#         prompts = []
#         responses = []

#         for row in rows:
#             prompts.append(row["prompt"])
#             responses.append(row["response"])

#         batch = build_sft_batch(tokenizer, prompts, responses, SFT_MAX_LENGTH)

#         with torch.no_grad():
#             with torch.autocast(device_type="cuda", dtype=torch.float16):
#                 response_logps = compute_response_logprobs(
#                     model, tokenizer, prompts, responses, SFT_MAX_LENGTH
#                 )

#         token_count = (batch["labels"][:, 1:] != -100).sum().item()
#         total_negative_logp += (-response_logps.sum()).item()
#         total_response_tokens += token_count

#     model.train()

#     if total_response_tokens == 0:
#         return 0.0

#     return total_negative_logp / total_response_tokens

In [16]:
# print("Loading UltraChat...")
# sft_train_dataset = load_dataset(
#     "HuggingFaceH4/ultrachat_200k", split="train_sft"
# ).shuffle(seed=SEED)

# sft_validation_dataset = load_dataset(
#     "HuggingFaceH4/ultrachat_200k", split="test_sft"
# ).shuffle(seed=SEED)

# sft_train_examples = make_sft_examples(sft_train_dataset, SFT_TRAIN_EXAMPLES)
# sft_validation_examples = make_sft_examples(sft_validation_dataset, SFT_VALIDATION_EXAMPLES)

# print("Train examples:", len(sft_train_examples))
# print("Validation examples:", len(sft_validation_examples))

In [17]:
# sft_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL)
# if sft_tokenizer.pad_token_id is None:
#     sft_tokenizer.pad_token = sft_tokenizer.eos_token

# sft_model = AutoModelForCausalLM.from_pretrained(
#     SFT_MODEL,
#     torch_dtype=torch.float32,
# ).to("cuda:0")
# sft_model.train()

# sft_optimizer = torch.optim.AdamW(
#     sft_model.parameters(),
#     lr=SFT_LEARNING_RATE,
#     weight_decay=SFT_WEIGHT_DECAY,
# )

# number_of_micro_batches = math.ceil(len(sft_train_examples) / SFT_BATCH_SIZE)
# optimizer_steps_per_epoch = math.ceil(number_of_micro_batches / SFT_GRAD_ACCUM)
# total_optimizer_steps = optimizer_steps_per_epoch * SFT_EPOCHS
# warmup_steps = int(total_optimizer_steps * SFT_WARMUP_RATIO)

# sft_scheduler = get_cosine_schedule_with_warmup(
#     sft_optimizer,
#     num_warmup_steps=warmup_steps,
#     num_training_steps=total_optimizer_steps,
# )

# sft_scaler = torch.cuda.amp.GradScaler()
# sft_optimizer.zero_grad()

In [18]:
# best_sft_validation_loss = float("inf")
# sft_optimizer_step = 0

# for epoch in range(SFT_EPOCHS):
#     accumulated_loss = 0.0
#     micro_batch_number = 0
#     remainder = number_of_micro_batches % SFT_GRAD_ACCUM

#     for start in range(0, len(sft_train_examples), SFT_BATCH_SIZE):
#         rows = sft_train_examples[start:start + SFT_BATCH_SIZE]
#         prompts = []
#         responses = []

#         for row in rows:
#             prompts.append(row["prompt"])
#             responses.append(row["response"])

#         with torch.autocast(device_type="cuda", dtype=torch.float16):
#             loss = compute_sft_loss(
#                 sft_model,
#                 sft_tokenizer,
#                 prompts,
#                 responses,
#                 SFT_MAX_LENGTH,
#             )

#         micro_batch_number += 1

#         accumulation_target = SFT_GRAD_ACCUM
#         if remainder != 0:
#             first_remainder_batch = number_of_micro_batches - remainder + 1
#             if micro_batch_number >= first_remainder_batch:
#                 accumulation_target = remainder

#         scaled_loss = loss / accumulation_target
#         sft_scaler.scale(scaled_loss).backward()
#         accumulated_loss += loss.item()

#         should_update = micro_batch_number % SFT_GRAD_ACCUM == 0
#         if micro_batch_number == number_of_micro_batches:
#             should_update = True

#         if should_update:
#             sft_scaler.unscale_(sft_optimizer)
#             torch.nn.utils.clip_grad_norm_(sft_model.parameters(), SFT_GRAD_CLIP)
#             sft_scaler.step(sft_optimizer)
#             sft_scaler.update()
#             sft_scheduler.step()
#             sft_optimizer.zero_grad()

#             sft_optimizer_step += 1
#             average_loss = accumulated_loss / accumulation_target
#             accumulated_loss = 0.0

#             save_jsonl_metric(
#                 SFT_METRICS,
#                 {
#                     "type": "train",
#                     "epoch": epoch + 1,
#                     "step": sft_optimizer_step,
#                     "loss": average_loss,
#                     "learning_rate": sft_scheduler.get_last_lr()[0],
#                 },
#             )

#     validation_loss = get_sft_validation_loss(
#         sft_model, sft_tokenizer, sft_validation_examples, "cuda:0"
#     )

#     save_jsonl_metric(
#         SFT_METRICS,
#         {
#             "type": "validation",
#             "epoch": epoch + 1,
#             "step": sft_optimizer_step,
#             "loss": validation_loss,
#         },
#     )

#     print("Validation loss:", validation_loss)

#     if validation_loss < best_sft_validation_loss:
#         best_sft_validation_loss = validation_loss
#         sft_model.save_pretrained(os.path.join(SFT_OUTPUT, "best"))
#         sft_tokenizer.save_pretrained(os.path.join(SFT_OUTPUT, "best"))
#         print("Saved best SFT checkpoint")

# sft_model.save_pretrained(os.path.join(SFT_OUTPUT, "final"))
# sft_tokenizer.save_pretrained(os.path.join(SFT_OUTPUT, "final"))
# print("Saved final SFT checkpoint")

In [19]:
# sft_train_steps = []
# sft_train_losses = []
# sft_val_steps = []
# sft_val_losses = []

# with open(SFT_METRICS, "r", encoding="utf-8") as file:
#     for line in file:
#         row = json.loads(line)
#         if row["type"] == "train":
#             sft_train_steps.append(row["step"])
#             sft_train_losses.append(row["loss"])
#         if row["type"] == "validation":
#             sft_val_steps.append(row["step"])
#             sft_val_losses.append(row["loss"])

# plt.figure(figsize=(8, 5))
# plt.plot(sft_train_steps, sft_train_losses, label="training loss")
# plt.scatter(sft_val_steps, sft_val_losses, label="validation loss")
# plt.xlabel("Optimizer step")
# plt.ylabel("Response-only loss")
# plt.title("SFT Training and Validation Loss")
# plt.legend()
# plt.tight_layout()
# plt.savefig("outputs/plots/sft_loss.png")
# plt.show()

In [20]:
# del sft_model
# del sft_optimizer
# del sft_scheduler
# del sft_scaler
# gc.collect()
# torch.cuda.empty_cache()

In [21]:
# DPO_TRAIN_PAIRS = 15000
# DPO_VALIDATION_PAIRS = 1000
# DPO_MAX_LENGTH = 512
# DPO_MAX_PROMPT_LENGTH = 256
# DPO_BATCH_SIZE = 1
# DPO_GRAD_ACCUM = 32
# DPO_LEARNING_RATE = 1e-6
# DPO_BETA = 0.1
# DPO_GRAD_CLIP = 1.0
# DPO_EPOCHS = 1
# SFT_BEST = "outputs/sft/best"

In [22]:
# def shorten_prompt(tokenizer, instruction):
#     empty_template = "### Instruction:\n\n\n### Response:\n"
#     template_ids = tokenizer.encode(empty_template, add_special_tokens=False)
#     instruction_ids = tokenizer.encode(instruction.strip(), add_special_tokens=False)

#     allowed = DPO_MAX_PROMPT_LENGTH - len(template_ids)
#     if allowed < 0:
#         allowed = 0

#     instruction_ids = instruction_ids[:allowed]
#     short_instruction = tokenizer.decode(instruction_ids, skip_special_tokens=True)
#     return "### Instruction:\n" + short_instruction.strip() + "\n\n### Response:\n"


# def parse_hh_text(text):
#     human_marker = "\n\nHuman:"
#     assistant_marker = "\n\nAssistant:"

#     if text.count(human_marker) != 1:
#         return None
#     if text.count(assistant_marker) != 1:
#         return None

#     after_human = text.split(human_marker, 1)[1]
#     parts = after_human.split(assistant_marker, 1)
#     return parts[0].strip(), parts[1].strip()


# def load_hh_examples(split, number_of_examples, tokenizer):
#     dataset = load_dataset("Anthropic/hh-rlhf", split=split).shuffle(seed=SEED)
#     examples = []

#     for row in dataset:
#         chosen_parts = parse_hh_text(row["chosen"])
#         rejected_parts = parse_hh_text(row["rejected"])

#         if chosen_parts is None or rejected_parts is None:
#             continue

#         chosen_prompt, chosen_response = chosen_parts
#         rejected_prompt, rejected_response = rejected_parts

#         if chosen_prompt != rejected_prompt:
#             continue

#         examples.append({
#             "prompt": shorten_prompt(tokenizer, chosen_prompt),
#             "chosen": chosen_response,
#             "rejected": rejected_response,
#         })

#         if len(examples) == number_of_examples:
#             break

#     return examples


# def load_ultrafeedback_examples(split, number_of_examples, tokenizer):
#     dataset = load_dataset(
#         "HuggingFaceH4/ultrafeedback_binarized", split=split
#     ).shuffle(seed=SEED)

#     examples = []
#     for index in range(number_of_examples):
#         row = dataset[index]
#         examples.append({
#             "prompt": shorten_prompt(tokenizer, row["prompt"]),
#             "chosen": row["chosen"][1]["content"],
#             "rejected": row["rejected"][1]["content"],
#         })

#     return examples

In [23]:
# def get_dpo_logps(model, tokenizer, prompts, chosen, rejected):
#     with torch.autocast(device_type="cuda", dtype=torch.float16):
#         chosen_logps = compute_response_logprobs(
#             model, tokenizer, prompts, chosen, DPO_MAX_LENGTH
#         )
#         rejected_logps = compute_response_logprobs(
#             model, tokenizer, prompts, rejected, DPO_MAX_LENGTH
#         )
#     return chosen_logps, rejected_logps


# def validate_dpo(policy, reference, tokenizer, examples):
#     policy.eval()
#     reference.eval()

#     total_loss = 0.0
#     correct = 0
#     total_margin = 0.0
#     total_examples = 0

#     for start in range(0, len(examples), DPO_BATCH_SIZE):
#         rows = examples[start:start + DPO_BATCH_SIZE]
#         prompts = []
#         chosen = []
#         rejected = []

#         for row in rows:
#             prompts.append(row["prompt"])
#             chosen.append(row["chosen"])
#             rejected.append(row["rejected"])

#         with torch.no_grad():
#             policy_chosen, policy_rejected = get_dpo_logps(
#                 policy, tokenizer, prompts, chosen, rejected
#             )
#             reference_chosen, reference_rejected = get_dpo_logps(
#                 reference, tokenizer, prompts, chosen, rejected
#             )

#         reference_chosen = reference_chosen.to(policy_chosen.device)
#         reference_rejected = reference_rejected.to(policy_chosen.device)

#         loss = compute_dpo_loss_from_logps(
#             policy_chosen,
#             policy_rejected,
#             reference_chosen,
#             reference_rejected,
#             DPO_BETA,
#         )

#         chosen_reward = DPO_BETA * (policy_chosen - reference_chosen)
#         rejected_reward = DPO_BETA * (policy_rejected - reference_rejected)
#         margins = chosen_reward - rejected_reward

#         total_loss += loss.item() * len(rows)
#         correct += (chosen_reward > rejected_reward).sum().item()
#         total_margin += margins.sum().item()
#         total_examples += len(rows)

#     policy.train()

#     validation_loss = total_loss / total_examples
#     reward_accuracy = correct / total_examples
#     reward_margin = total_margin / total_examples
#     return validation_loss, reward_accuracy, reward_margin


In [24]:
# def train_dpo(dataset_name, output_dir):
#     os.makedirs(output_dir, exist_ok=True)
#     metrics_path = os.path.join(output_dir, "metrics.jsonl")
#     if os.path.exists(metrics_path):
#         os.remove(metrics_path)

#     tokenizer = AutoTokenizer.from_pretrained(SFT_BEST)
#     if tokenizer.pad_token_id is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     policy = AutoModelForCausalLM.from_pretrained(
#         SFT_BEST,
#         torch_dtype=torch.float32,
#     ).to("cuda:0")
#     policy.train()

#     reference = AutoModelForCausalLM.from_pretrained(
#         SFT_BEST,
#         torch_dtype=torch.float16,
#     ).to("cuda:1")
#     reference.eval()

#     for parameter in reference.parameters():
#         parameter.requires_grad = False

#     if dataset_name == "hh":
#         train_examples = load_hh_examples("train", DPO_TRAIN_PAIRS, tokenizer)
#         validation_examples = load_hh_examples("test", DPO_VALIDATION_PAIRS, tokenizer)
#     else:
#         train_examples = load_ultrafeedback_examples(
#             "train_prefs", DPO_TRAIN_PAIRS, tokenizer
#         )
#         validation_examples = load_ultrafeedback_examples(
#             "test_prefs", DPO_VALIDATION_PAIRS, tokenizer
#         )

#     print(dataset_name, "training pairs:", len(train_examples))
#     print(dataset_name, "validation pairs:", len(validation_examples))

#     optimizer = torch.optim.RMSprop(policy.parameters(), lr=DPO_LEARNING_RATE)
#     scaler = torch.cuda.amp.GradScaler()
#     optimizer.zero_grad()

#     number_of_micro_batches = math.ceil(len(train_examples) / DPO_BATCH_SIZE)
#     remainder = number_of_micro_batches % DPO_GRAD_ACCUM
#     optimizer_step = 0
#     best_reward_accuracy = -1.0

#     for epoch in range(DPO_EPOCHS):
#         accumulated_loss = 0.0
#         micro_batch_number = 0

#         for start in range(0, len(train_examples), DPO_BATCH_SIZE):
#             rows = train_examples[start:start + DPO_BATCH_SIZE]
#             prompts = []
#             chosen = []
#             rejected = []

#             for row in rows:
#                 prompts.append(row["prompt"])
#                 chosen.append(row["chosen"])
#                 rejected.append(row["rejected"])

#             policy_chosen, policy_rejected = get_dpo_logps(
#                 policy, tokenizer, prompts, chosen, rejected
#             )

#             with torch.no_grad():
#                 reference_chosen, reference_rejected = get_dpo_logps(
#                     reference, tokenizer, prompts, chosen, rejected
#                 )

#             reference_chosen = reference_chosen.to(policy_chosen.device)
#             reference_rejected = reference_rejected.to(policy_chosen.device)

#             loss = compute_dpo_loss_from_logps(
#                 policy_chosen,
#                 policy_rejected,
#                 reference_chosen,
#                 reference_rejected,
#                 DPO_BETA,
#             )

#             micro_batch_number += 1

#             accumulation_target = DPO_GRAD_ACCUM
#             if remainder != 0:
#                 first_remainder_batch = number_of_micro_batches - remainder + 1
#                 if micro_batch_number >= first_remainder_batch:
#                     accumulation_target = remainder

#             scaled_loss = loss / accumulation_target
#             scaler.scale(scaled_loss).backward()
#             accumulated_loss += loss.item()

#             should_update = micro_batch_number % DPO_GRAD_ACCUM == 0
#             if micro_batch_number == number_of_micro_batches:
#                 should_update = True

#             if should_update:
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(policy.parameters(), DPO_GRAD_CLIP)
#                 scaler.step(optimizer)
#                 scaler.update()
#                 optimizer.zero_grad()

#                 optimizer_step += 1
#                 average_loss = accumulated_loss / accumulation_target
#                 accumulated_loss = 0.0

#                 save_jsonl_metric(
#                     metrics_path,
#                     {
#                         "type": "train",
#                         "epoch": epoch + 1,
#                         "step": optimizer_step,
#                         "dpo_loss": average_loss,
#                     },
#                 )

#         validation_loss, reward_accuracy, reward_margin = validate_dpo(
#             policy, reference, tokenizer, validation_examples
#         )

#         save_jsonl_metric(
#             metrics_path,
#             {
#                 "type": "validation",
#                 "epoch": epoch + 1,
#                 "step": optimizer_step,
#                 "dpo_loss": validation_loss,
#                 "reward_accuracy": reward_accuracy,
#                 "reward_margin": reward_margin,
#             },
#         )

#         print(dataset_name, "validation DPO loss:", validation_loss)
#         print(dataset_name, "reward accuracy:", reward_accuracy)
#         print(dataset_name, "reward margin:", reward_margin)

#         if reward_accuracy > best_reward_accuracy:
#             best_reward_accuracy = reward_accuracy
#             policy.save_pretrained(os.path.join(output_dir, "best"))
#             tokenizer.save_pretrained(os.path.join(output_dir, "best"))
#             print("Saved best", dataset_name, "DPO checkpoint")

#     policy.save_pretrained(os.path.join(output_dir, "final"))
#     tokenizer.save_pretrained(os.path.join(output_dir, "final"))
#     print("Saved final", dataset_name, "DPO checkpoint")

#     del policy
#     del reference
#     del optimizer
#     del scaler
#     gc.collect()
#     torch.cuda.empty_cache()

In [25]:
# train_dpo("hh", "outputs/dpo_hh")

In [26]:
# train_dpo("ultrafeedback", "outputs/dpo_ultrafeedback")

In [27]:
# def plot_dpo_metrics(metrics_path, title, output_path):
#     train_steps = []
#     train_losses = []
#     val_steps = []
#     val_losses = []
#     reward_accuracies = []
#     reward_margins = []

#     with open(metrics_path, "r", encoding="utf-8") as file:
#         for line in file:
#             row = json.loads(line)
#             if row["type"] == "train":
#                 train_steps.append(row["step"])
#                 train_losses.append(row["dpo_loss"])
#             if row["type"] == "validation":
#                 val_steps.append(row["step"])
#                 val_losses.append(row["dpo_loss"])
#                 reward_accuracies.append(row["reward_accuracy"])
#                 reward_margins.append(row["reward_margin"])

#     plt.figure(figsize=(8, 5))
#     plt.plot(train_steps, train_losses, label="training DPO loss")
#     plt.scatter(val_steps, val_losses, label="validation DPO loss")
#     plt.xlabel("Optimizer step")
#     plt.ylabel("DPO loss")
#     plt.title(title + " - DPO Loss")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(output_path + "_loss.png")
#     plt.show()

#     plt.figure(figsize=(8, 5))
#     plt.scatter(val_steps, reward_accuracies, label="reward accuracy")
#     plt.scatter(val_steps, reward_margins, label="reward margin")
#     plt.xlabel("Optimizer step")
#     plt.ylabel("Validation value")
#     plt.title(title + " - Preference Diagnostics")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(output_path + "_preference.png")
#     plt.show()

# plot_dpo_metrics(
#     "outputs/dpo_hh/metrics.jsonl",
#     "HH-RLHF DPO",
#     "outputs/plots/dpo_hh",
# )

# plot_dpo_metrics(
#     "outputs/dpo_ultrafeedback/metrics.jsonl",
#     "UltraFeedback DPO",
#     "outputs/plots/dpo_ultrafeedback",
# )

In [28]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

PERSONA_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
PERSONA_DATA = "data/persona_lora/ramsay_style_preferences.json"
PERSONA_OUTPUT = "outputs/persona_lora"

with open(PERSONA_DATA, "r", encoding="utf-8") as file:
    raw_persona_rows = json.load(file)

formatted_persona_rows = []
for row in raw_persona_rows:
    formatted_persona_rows.append({
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "chosen": [{"role": "assistant", "content": row["chosen"]}],
        "rejected": [{"role": "assistant", "content": row["rejected"]}],
    })

persona_dataset = Dataset.from_list(formatted_persona_rows)
print("Persona preference pairs:", len(persona_dataset))

Persona preference pairs: 220


In [29]:
persona_tokenizer = AutoTokenizer.from_pretrained(PERSONA_MODEL)
if persona_tokenizer.pad_token_id is None:
    persona_tokenizer.pad_token = persona_tokenizer.eos_token
persona_tokenizer.padding_side = "left"

persona_model = AutoModelForCausalLM.from_pretrained(
    PERSONA_MODEL,
    torch_dtype=torch.float16,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

persona_args = DPOConfig(
    output_dir=PERSONA_OUTPUT,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    fp16=True,
    seed=SEED,
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=1,
    report_to="none",
)

persona_trainer = DPOTrainer(
    model=persona_model,
    args=persona_args,
    train_dataset=persona_dataset,
    processing_class=persona_tokenizer,
    peft_config=lora_config,
)

persona_trainer.train()

persona_final = os.path.join(PERSONA_OUTPUT, "final")
persona_trainer.save_model(persona_final)
persona_tokenizer.save_pretrained(persona_final)

with open(os.path.join(PERSONA_OUTPUT, "metrics.json"), "w", encoding="utf-8") as file:
    json.dump(persona_trainer.state.log_history, file, indent=2)

print("Saved LoRA-DPO adapter to", persona_final)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Applying chat template to train dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
1,0.693100
2,0.692000
3,0.688800
4,0.686800
5,0.682600
6,0.680400
7,0.673500
8,0.675600
9,0.668000
10,0.670600


Saved LoRA-DPO adapter to outputs/persona_lora/final
